# Train MuJoCo Playground's SO-101 PickCube for mjswan

MuJoCo Playground 1.0.1's `SO101PickCube`, the state-observation sibling of the task its
sim-to-real demo deploys, trained with Playground's own environment and PPO settings, then
exported as the ONNX policy an mjswan scene runs.

1. **Runtime → Change runtime type → T4 GPU**, then run the cells in order.
2. Everything is kept in your Google Drive (`OUT`), so a disconnected session resumes from the
   last checkpoint when you run the cells again.
3. The result is `OUT/export`: `policy.onnx`, `contract.json` and `reference.npz`
   (see [`so101_pick_cube.py`](https://github.com/ttktjmt/mjswan/blob/main/examples/train/so101_pick_cube.py)).

In [ ]:
!nvidia-smi

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")

OUT = "/content/drive/MyDrive/mjswan/so101_pick_cube"
ITERATIONS = 1000
# Optional: a Hugging Face model repository ("user/name") that also keeps every checkpoint.
# Add your token as the Colab secret HF_TOKEN first.
HF_REPO = ""
# The mjswan branch the training script is fetched from.
BRANCH = "main"

HUB = f"--hf-repo {HF_REPO}" if HF_REPO else ""
if HF_REPO:
    from google.colab import userdata

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
!wget -q -O so101_pick_cube.py https://raw.githubusercontent.com/ttktjmt/mjswan/{BRANCH}/examples/train/so101_pick_cube.py
!python so101_pick_cube.py setup

Training curves while it runs (optional).

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "$OUT/checkpoints"

Train. On a T4 this takes a while; if the session disconnects, run the cells above again
and then this one: it resumes from the newest checkpoint in `OUT/checkpoints`.

In [ ]:
!python so101_pick_cube.py train --out "{OUT}" --iterations {ITERATIONS} {HUB}

In [ ]:
!python so101_pick_cube.py export --out "{OUT}"
!python so101_pick_cube.py upload --out "{OUT}" {HUB}

In [ ]:
import shutil

from google.colab import files

archive = shutil.make_archive("/content/so101_pick_cube_export", "zip", f"{OUT}/export")
files.download(archive)